In [1]:
from superfv.tools.stability_analysis.fv_advection import derive_fv_upwind_rhs
from superfv.tools.stability_analysis.fv_polynomials import get_index_from_U_i
from superfv.tools.stability_analysis.sd_advection import sp_to_fp_matrix_symbolic, fp_to_sp_prime_matrix_symbolic
from superfv.tools.stability_analysis.variables import a, h 
import numpy as np
import sympy as sp

p = 2
N_elems = 4
N_dofs = N_elems * (p + 1)
N_dofs

12

Derive FV spatial operator as matrix

In [2]:
derive_fv_upwind_rhs(p)

U_-1*a/h - U_-2*a/(6*h) - U_0*a/(2*h) - U_1*a/(3*h)

In [3]:
stencil = sp.expand(derive_fv_upwind_rhs(p) * h / a)
min_idx = min(get_index_from_U_i(term) for term in stencil.as_ordered_terms())
max_idx = max(get_index_from_U_i(term) for term in stencil.as_ordered_terms())

band_weights = {i: stencil.coeff(sp.Symbol(f"U_{i}", real=True)) for i in range(min_idx, max_idx + 1)}
band_weights

{-2: -1/6, -1: 1, 0: -1/2, 1: -1/3}

In [4]:
fv_matrix = np.zeros((N_dofs, N_dofs), dtype=object)

for i in range(fv_matrix.shape[0]):
    for j, weight in band_weights.items():
        fv_matrix[i, (i + j) % N_dofs] = weight

sp.simplify(fv_matrix)

[[-1/2, -1/3, 0, 0, 0, 0, 0, 0, 0, 0, -1/6, 1], [1, -1/2, -1/3, 0, 0, 0, 0, 0, 0, 0, 0, -1/6], [-1/6, 1, -1/2, -1/3, 0, 0, 0, 0, 0, 0, 0, 0], [0, -1/6, 1, -1/2, -1/3, 0, 0, 0, 0, 0, 0, 0], [0, 0, -1/6, 1, -1/2, -1/3, 0, 0, 0, 0, 0, 0], [0, 0, 0, -1/6, 1, -1/2, -1/3, 0, 0, 0, 0, 0], [0, 0, 0, 0, -1/6, 1, -1/2, -1/3, 0, 0, 0, 0], [0, 0, 0, 0, 0, -1/6, 1, -1/2, -1/3, 0, 0, 0], [0, 0, 0, 0, 0, 0, -1/6, 1, -1/2, -1/3, 0, 0], [0, 0, 0, 0, 0, 0, 0, -1/6, 1, -1/2, -1/3, 0], [0, 0, 0, 0, 0, 0, 0, 0, -1/6, 1, -1/2, -1/3], [-1/3, 0, 0, 0, 0, 0, 0, 0, 0, -1/6, 1, -1/2]]

Derive SD spatial operator as two matrices

In [5]:
sp.simplify(sp_to_fp_matrix_symbolic(p))

[[2/3 - sqrt(3)/3, -1/3, sqrt(3)/3 + 2/3], [-1/9, 5/9, 5/9], [5/9, 5/9, -1/9], [sqrt(3)/3 + 2/3, -1/3, 2/3 - sqrt(3)/3]]

In [6]:
sp_matrix = sp_to_fp_matrix_symbolic(p)
periodic_sp_matrix = sp_matrix.copy()
periodic_sp_matrix[0] = 0

sd_matrix1 = np.zeros((N_elems * (p + 2), N_dofs), dtype=object)
for m in range(N_elems):
    rowslc = slice(m * (p+2), (m+1) * (p+2))
    colslc = slice(m * (p+1), (m+1) * (p+1))
    sd_matrix1[(rowslc, colslc)] = periodic_sp_matrix

    mleft = (m - 1) % N_elems
    colslc_left = slice(mleft * (p+1), (mleft+1) * (p+1))
    sd_matrix1[m * (p+2), colslc_left] = sp_matrix[-1]

sp.simplify(sd_matrix1)

[[0, 0, 0, 0, 0, 0, 0, 0, 0, sqrt(3)/3 + 2/3, -1/3, 2/3 - sqrt(3)/3], [-1/9, 5/9, 5/9, 0, 0, 0, 0, 0, 0, 0, 0, 0], [5/9, 5/9, -1/9, 0, 0, 0, 0, 0, 0, 0, 0, 0], [sqrt(3)/3 + 2/3, -1/3, 2/3 - sqrt(3)/3, 0, 0, 0, 0, 0, 0, 0, 0, 0], [sqrt(3)/3 + 2/3, -1/3, 2/3 - sqrt(3)/3, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, -1/9, 5/9, 5/9, 0, 0, 0, 0, 0, 0], [0, 0, 0, 5/9, 5/9, -1/9, 0, 0, 0, 0, 0, 0], [0, 0, 0, sqrt(3)/3 + 2/3, -1/3, 2/3 - sqrt(3)/3, 0, 0, 0, 0, 0, 0], [0, 0, 0, sqrt(3)/3 + 2/3, -1/3, 2/3 - sqrt(3)/3, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, -1/9, 5/9, 5/9, 0, 0, 0], [0, 0, 0, 0, 0, 0, 5/9, 5/9, -1/9, 0, 0, 0], [0, 0, 0, 0, 0, 0, sqrt(3)/3 + 2/3, -1/3, 2/3 - sqrt(3)/3, 0, 0, 0], [0, 0, 0, 0, 0, 0, sqrt(3)/3 + 2/3, -1/3, 2/3 - sqrt(3)/3, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, -1/9, 5/9, 5/9], [0, 0, 0, 0, 0, 0, 0, 0, 0, 5/9, 5/9, -1/9], [0, 0, 0, 0, 0, 0, 0, 0, 0, sqrt(3)/3 + 2/3, -1/3, 2/3 - sqrt(3)/3]]

In [7]:
sp_prime_matrix = fp_to_sp_prime_matrix_symbolic(p)

sd_matrix2 = np.zeros((N_dofs, N_elems * (p + 2)), dtype=object)
for m in range(N_elems):
    rowslc = slice(m * (p+1), (m+1) * (p+1))
    colslc = slice(m * (p+2), (m+1) * (p+2))
    sd_matrix2[(rowslc, colslc)] = sp_prime_matrix

sp.simplify(sd_matrix2)

[[-23/24 + sqrt(3)/2, sqrt(3)/8, -9*sqrt(3)/8, sqrt(3)/2 + 23/24, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [1/6, -sqrt(3)/2, sqrt(3)/2, -1/6, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [-23/24 - sqrt(3)/2, 9*sqrt(3)/8, -sqrt(3)/8, 23/24 - sqrt(3)/2, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, -23/24 + sqrt(3)/2, sqrt(3)/8, -9*sqrt(3)/8, sqrt(3)/2 + 23/24, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 1/6, -sqrt(3)/2, sqrt(3)/2, -1/6, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, -23/24 - sqrt(3)/2, 9*sqrt(3)/8, -sqrt(3)/8, 23/24 - sqrt(3)/2, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, -23/24 + sqrt(3)/2, sqrt(3)/8, -9*sqrt(3)/8, sqrt(3)/2 + 23/24, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 1/6, -sqrt(3)/2, sqrt(3)/2, -1/6, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, -23/24 - sqrt(3)/2, 9*sqrt(3)/8, -sqrt(3)/8, 23/24 - sqrt(3)/2, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, -23/24 + sqrt(3)/2, sqrt(3)/8, -9*sqrt(3)/8, sqrt(3)/2 + 23/24], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1/6, -sqrt(3)/2, sqrt(3)/2, -1/6